# EDA de la population en incident de paiement

**Objectif** : définir une règle métier qui repère les clients en **incident de paiement** (deux codifications de retard d'affilée, ou une codification de 3 ou plus), afin de :
- les sortir du périmètre du ML et les traiter par une règle métier (prédits en défaut),
- entraîner les modèles sur une population au comportement plus homogène.

Une seconde population sort aussi du ML, sans être mêlée à la première : le **retard imminent** (un seul retard, posé par la banque en septembre après un mois d'août sain).

**Vocabulaire** : un **retard** est un mois codifié 2 ou plus ; un **incident de paiement** est un retard qui dure (deux mois d'affilée) ou qui est grave (3 ou plus) ; un **retard régularisé** est un retard isolé dont le client est sorti.

**Méthodologie**
- Les règles sont définies **a priori, par logique métier**, à partir des constats de l'EDA lab. Aucun seuil n'est optimisé sur la cible.
- Le jeu est découpé **avant toute analyse** en train / test (80/20, stratifié sur `dpnm`, `random_state=42`). Toutes les analyses sont faites sur le **train**.
- Les règles sont d'abord décrites par leur **comportement** (volumes, paiements, dette), sans `dpnm`.
- `dpnm` n'est utilisé sur le train que pour **valider** qu'un périmètre a un sens métier. Le **test** n'est utilisé **qu'une seule fois**, à la fin, pour la règle retenue.

Données : cleaned2_creditcard.csv (niveau 2 du nettoyage : suppressions du niveau 1, puis codifications 1 corrigées ; la codification de la banque est gardée dans PAY_1_original), restreint au périmètre S12 (BILL_AMT1 > 0)

**Lecture des colonnes** : `PAY_n` juge la facture de fin du mois n (`BILL_AMTn`) sur le paiement fait le mois suivant (`PAY_AMT(n-1)`) ; preuve dans `01_01_audit` (partie « Temporalité des colonnes »).

**Plan** : définition retenue et statuts en section 4, validation sur le test en section 5, indicateurs pour le ML en section 6, mise à l'épreuve sur le dataset initial en section 7. En annexe : l'étude des codifications de retard qui justifie la population, les règles exploratoires, puis la sortie du retard et les retards d'avril.

## 1. Chargement et périmètre

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Retrouve la racine du projet (là où se trouve le dossier 'data')
current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())

df = pd.read_csv(root_dir / "data" / "cleaned2_creditcard.csv")
print(f"Dataset complet : {len(df)} lignes | taux de défaut : {df['dpnm'].mean() * 100:.2f}%")

# Périmètre S12, avant le découpage (règle du projet) : dette en septembre et plafond de 500 000 NT$ au plus
df = df[(df['BILL_AMT1'] > 0) & (df['LIMIT_BAL'] <= 500000)].copy()
print(f"Périmètre S12 : {len(df)} lignes | taux de défaut : {df['dpnm'].mean() * 100:.2f}%")

pay_cols = [f'PAY_{i}' for i in range(1, 7)]
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_amt_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

Dataset complet : 28851 lignes | taux de défaut : 21.74%
Périmètre S12 : 27202 lignes | taux de défaut : 21.95%


## 2. Séparation train / test
Le split est fait avant toute analyse et ne dépend d'aucune définition de la population en incident. Les notebooks ML devront reproduire ce même split (même périmètre, `stratify=dpnm`, `random_state=42`).

In [2]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['dpnm'], random_state=42)

print(f"Train : {len(df_train)} lignes | taux de défaut : {df_train['dpnm'].mean() * 100:.2f}%")
print(f"Test  : {len(df_test)} lignes | taux de défaut : {df_test['dpnm'].mean() * 100:.2f}%")

Train : 21761 lignes | taux de défaut : 21.95%
Test  : 5441 lignes | taux de défaut : 21.96%


## 3. Fonctions de description et de validation
- `profil_comportement` : comportement d'un groupe de clients sans la cible (mois sans paiement, évolution de la dette, utilisation du plafond, plafond) ;
- `validation` : pour un périmètre, taux de défaut, part des défauts captés et taux de défaut du reste (train seulement).

In [3]:
def profil_comportement(d):
    # Nombre de mois sans aucun paiement
    mois_sans_paiement = (d[pay_amt_cols] == 0).sum(axis=1)
    # Évolution de la dette de M-6 à M-1
    evolution_dette = d['BILL_AMT1'] - d['BILL_AMT6']
    # Utilisation du plafond à M-1
    utilisation = d['BILL_AMT1'] / d['LIMIT_BAL'] * 100
    return pd.Series({
        'Nb clients': len(d),
        'Mois sans paiement (moy.)': mois_sans_paiement.mean(),
        '% avec 6 mois sans paiement': (mois_sans_paiement == 6).mean() * 100,
        '% dette en hausse M-6 -> M-1': (evolution_dette > 0).mean() * 100,
        'Utilisation plafond M-1 (médiane %)': utilisation.median(),
        'LIMIT_BAL (médiane)': d['LIMIT_BAL'].median(),
    })

def validation(d, masque):
    y = d['dpnm']
    return pd.Series({
        'Nb clients': masque.sum(),
        'Part (%)': masque.mean() * 100,
        'Taux défaut périmètre (%)': y[masque].mean() * 100,
        'Défauts captés (%)': y[masque].sum() / y.sum() * 100,
        'Taux défaut reste (%)': y[~masque].mean() * 100,
    })

print(f"Taux de défaut global (train) : {df_train['dpnm'].mean() * 100:.2f}%")

Taux de défaut global (train) : 21.95%


## 4. Définition retenue : deux codifications de retard successives, ou une codification de 3 ou plus

### 4.1 Règle 1 : la définition sur les codifications telles quelles

**Un passage en incident de paiement, c'est deux codifications de retard (>= 2) successives, ou une seule codification >= 3** (un retard ne monte que d'un cran par mois : un 3 suppose déjà deux mois de retard). Les codifications sont lues telles quelles, avec `PAY_1` corrigé au niveau 2 du nettoyage.

| Élément | Règle |
|---|---|
| Historique (mai à août) | série d'au moins 2 mois à >= 2, ou codification >= 3 : **passage en incident** (`Sorti de l'incident`, `FLAG_INCIDENT`, `MOIS_SORTIE_INCIDENT`) ; 2 isolé : **retard régularisé** (`FLAG_RETARD`, `MOIS_SORTIE_RETARD` = mois du retour sous 2, de 1 à 5) |
| Avril (M-6), mars inconnu | codification >= 3 : passage en incident ; 2 isolé : **simple retard** (retard régularisé) si le paiement d'avril couvre la facture de mai (`PAY_AMT6` >= `BILL_AMT5`), sinon passage en incident par défaut |
| Septembre (M-1) | `PAY_1` et `PAY_2` >= 2, ou `PAY_1` >= 3 : **en incident à M** ; `PAY_1` = 2 avec août sain : **retard imminent** (`FLAG_RETARD_IMMINENT` = 1) |

Statuts à M : `En incident` et `Retard imminent` (retirés du ML, chacun comme une population à part), `Sorti de l'incident`, `Retard régularisé`, `Jamais en retard` (dans le ML).

**Pourquoi le retard imminent sort du ML** : `PAY_1` = 2 veut dire que la banque a déjà jugé la facture de septembre insuffisamment payée en octobre. Le client est en retard à M selon la banque elle-même ; un modèle n'a rien à prédire pour lui. C'est la seule situation, hors incident, où le retard n'est pas régularisé.

`FLAG_INCIDENT` = 1 si le client est passé par un incident de paiement à un moment des six mois, ou s'y trouve à M. `MOIS_SORTIE_INCIDENT` = mois de la dernière sortie (1 à 5), -1 s'il n'y a pas de sortie (client encore en incident, ou jamais en incident). `FLAG_RETARD` = 1 si le client a eu un retard isolé régularisé ; `MOIS_SORTIE_RETARD` = mois du retour sous 2 (1 à 5), -1 sinon. **Sont retirés du dataset ML les clients en incident à M (`FLAG_INCIDENT` = 1 et `MOIS_SORTIE_INCIDENT` = -1) et les clients en retard imminent (`FLAG_RETARD_IMMINENT` = 1).** Un client en retard imminent garde son historique (passage en incident ou retard régularisé plus anciens) dans les autres colonnes.

In [4]:
def statut_incident(d):
    # Codifications lues telles quelles (PAY_1 corrigé au niveau 2 du nettoyage)
    codes = d[pay_cols].copy()
    niveaux = codes.values
    # Avril, mars inconnu : un 2 isolé n'est qu'un simple retard si le paiement d'avril couvre la facture de mai
    simple_retard_avril = (d['PAY_AMT6'] >= d['BILL_AMT5']).values

    # Historique : série d'au moins 2 mois à >= 2, ou codification >= 3 = passage en incident ; 2 isolé = retard régularisé
    retard = niveaux >= 2
    passage_incident = np.zeros(len(d), bool)
    retard_regularise = np.zeros(len(d), bool)
    mois_sortie = np.full(len(d), -1.0)
    mois_regularisation = np.full(len(d), -1.0)
    for i, r in enumerate(retard):
        k = 1
        while k < 6:
            if r[k] and not r[k - 1]:
                fin = k
                while fin + 1 < 6 and r[fin + 1]:
                    fin += 1
                # k = mois du retour sous 2 (1 = M-1), séries parcourues de la plus récente à la plus ancienne
                serie = fin > k or niveaux[i, k:fin + 1].max() >= 3
                if fin == 5 and not serie:
                    serie = not simple_retard_avril[i]   # 2 isolé en avril : incident sauf simple retard
                if serie:
                    if not passage_incident[i]:
                        mois_sortie[i] = k
                    passage_incident[i] = True
                else:
                    if not retard_regularise[i]:
                        mois_regularisation[i] = k
                    retard_regularise[i] = True
                k = fin + 1
            else:
                k += 1
    passage_incident = pd.Series(passage_incident, index=d.index)
    retard_regularise = pd.Series(retard_regularise, index=d.index)

    # Septembre : en incident à M si deux retards d'affilée (septembre et août) ou un 3 ; un 2 seul = retard imminent
    incident_m = ((codes['PAY_1'] >= 2) & (codes['PAY_2'] >= 2)) | (codes['PAY_1'] >= 3)
    retard_imminent = (codes['PAY_1'] >= 2) & ~incident_m

    statut = pd.Series(np.select(
        [incident_m, retard_imminent, passage_incident, retard_regularise],
        ['En incident', 'Retard imminent', "Sorti de l'incident", 'Retard régularisé'],
        default='Jamais en retard',
    ), index=d.index)

    # En incident à M = FLAG_INCIDENT = 1 et MOIS_SORTIE_INCIDENT = -1 (retiré du dataset ML)
    mois_sortie = pd.Series(mois_sortie, index=d.index)
    mois_sortie[incident_m] = -1
    mois_regularisation = pd.Series(mois_regularisation, index=d.index)
    mois_regularisation[incident_m] = -1

    return pd.DataFrame({
        'STATUT_INCIDENT': statut,
        'MOIS_SORTIE_INCIDENT': mois_sortie,
        'FLAG_INCIDENT': (passage_incident | incident_m).astype(int),
        'MOIS_SORTIE_RETARD': mois_regularisation,
        'FLAG_RETARD': (retard_regularise & ~incident_m).astype(int),
        'FLAG_RETARD_IMMINENT': retard_imminent.astype(int),
    })


def retires_du_ml(f):
    # Les deux populations retirées du ML, gardées séparées
    return pd.DataFrame({
        'En incident à M': (f['FLAG_INCIDENT'] == 1) & (f['MOIS_SORTIE_INCIDENT'] == -1),
        'Retard imminent': f['FLAG_RETARD_IMMINENT'] == 1,
    })


final2_train = statut_incident(df_train)
statuts_r2 = ['En incident', 'Retard imminent', "Sorti de l'incident", 'Retard régularisé', 'Jamais en retard']

print("Statut à M (définition 2) :")
display(final2_train['STATUT_INCIDENT'].value_counts().reindex(statuts_r2).rename_axis('Statut à M').to_frame('Nb clients'))

# Contrôles : chaque population retirée correspond exactement à son statut, et les deux ne se recouvrent pas
retires = retires_du_ml(final2_train)
print(f"Retirés du dataset ML : en incident à M {retires['En incident à M'].sum()} "
      f"(identiques au statut : {(retires['En incident à M'] == (final2_train['STATUT_INCIDENT'] == 'En incident')).all()}) ; "
      f"retard imminent {retires['Retard imminent'].sum()} "
      f"(identiques au statut : {(retires['Retard imminent'] == (final2_train['STATUT_INCIDENT'] == 'Retard imminent')).all()}) ; "
      f"recouvrement : {int((retires['En incident à M'] & retires['Retard imminent']).sum())}")

print("Comportement par statut (sans la cible) :")
display(pd.DataFrame({s: profil_comportement(df_train[final2_train['STATUT_INCIDENT'] == s]) for s in statuts_r2}).T.round(1).rename_axis('Statut à M'))

Statut à M (définition 2) :


,Nb clients
Statut à M,
En incident,3019
Retard imminent,779
Sorti de l'incident,1373
Retard régularisé,1397
Jamais en retard,15193


Retirés du dataset ML : en incident à M 3019 (identiques au statut : True) ; retard imminent 779 (identiques au statut : True) ; recouvrement : 0
Comportement par statut (sans la cible) :


,Nb clients,Mois sans paiement (moy.),% avec 6 mois sans paiement,% dette en hausse M-6 -> M-1,Utilisation plafond M-1 (médiane %),LIMIT_BAL (médiane)
Statut à M,,,,,,
En incident,3019.0,1.9,5.4,43.1,74.3,60000.0
Retard imminent,779.0,0.8,0.0,55.6,74.1,80000.0
Sorti de l'incident,1373.0,1.8,2.3,41.6,43.1,80000.0
Retard régularisé,1397.0,1.4,0.0,54.2,24.3,100000.0
Jamais en retard,15193.0,0.7,1.7,67.0,29.8,160000.0


In [5]:
print("Validation par statut à M (définition 2) :")
display(pd.DataFrame({s: validation(df_train, final2_train['STATUT_INCIDENT'] == s) for s in statuts_r2}).T.round(2).rename_axis('Statut à M'))

# Durée du retard en cours à M, pour la population en incident : six mois d'affilée, ou deux à cinq mois
en_incident = final2_train['STATUT_INCIDENT'] == 'En incident'
duree = (df_train[pay_cols] >= 2).cumprod(axis=1).sum(axis=1)
print("Détail de la population en incident, selon la durée du retard en cours à M :")
display(pd.DataFrame({'Retard depuis 6 mois': validation(df_train, en_incident & (duree == 6)),
                      'Retard depuis 2 à 5 mois': validation(df_train, en_incident & (duree < 6))}).T.round(2).rename_axis('Durée'))

sortis2 = final2_train['STATUT_INCIDENT'] == "Sorti de l'incident"
print("Sortis de l'incident selon le mois de sortie :")
display(df_train[sortis2].groupby(final2_train.loc[sortis2, 'MOIS_SORTIE_INCIDENT'])['dpnm']
        .agg(Nb='count', Taux_defaut=lambda x: x.mean() * 100).round(1))

regul2 = final2_train['STATUT_INCIDENT'] == 'Retard régularisé'
print("Retards régularisés selon le mois de retour sous 2 :")
display(df_train[regul2].groupby(final2_train.loc[regul2, 'MOIS_SORTIE_RETARD'])['dpnm']
        .agg(Nb='count', Taux_defaut=lambda x: x.mean() * 100).round(1))

Validation par statut à M (définition 2) :


,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Statut à M,,,,,
En incident,3019.0,13.87,58.93,37.24,16.00
Retard imminent,779.0,3.58,65.08,10.61,20.35
Sorti de l'incident,1373.0,6.31,30.44,8.75,21.38
Retard régularisé,1397.0,6.42,23.55,6.89,21.84
Jamais en retard,15193.0,69.82,11.48,36.51,46.18


Détail de la population en incident, selon la durée du retard en cours à M :


,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Durée,,,,,
Retard depuis 6 mois,1095.0,5.03,70.50,16.16,19.38
Retard depuis 2 à 5 mois,1924.0,8.84,52.34,21.08,19.00


Sortis de l'incident selon le mois de sortie :


,Nb,Taux_defaut
MOIS_SORTIE_INCIDENT,,
1.0,213,39.4
2.0,333,34.2
3.0,221,34.4
4.0,134,27.6
5.0,472,22.7


Retards régularisés selon le mois de retour sous 2 :


,Nb,Taux_defaut
MOIS_SORTIE_RETARD,,
1.0,277,31.8
2.0,531,23.2
3.0,363,21.8
4.0,202,18.3
5.0,24,8.3


**Interprétation**
- Les statuts gardent une gradation du risque : la population en incident à M fait bien plus défaut que les sortis de l'incident, eux-mêmes plus que les retards régularisés, eux-mêmes plus que les clients jamais en retard.
- Le retard imminent (un 2 en septembre après un août sain) fait défaut au moins aussi souvent que la population en incident : la banque l'a déjà mis en retard. Il sort du ML, comme population à part : c'est un retard d'un seul mois, pas un incident.
- Dans la population en incident, un retard en cours depuis six mois est plus risqué qu'un retard de deux à cinq mois.
- Un vrai passage en incident (au moins deux mois à >= 2, ou un 3) et un retard isolé régularisé ont des risques nettement différents ; dans chacun des deux groupes, le risque baisse quand le retour sous 2 est plus ancien.
- En avril, un 2 isolé n'est un simple retard que si le paiement d'avril couvre la facture de mai ; sinon il compte comme un passage en incident, faute de voir mars.

### 4.2 Règle 2 : uniformisation des codifications 2 d'avril à juin, puis la même règle

En avril, mai et juin, la banque laissait sains une partie des clients qui ne payaient presque rien ; à partir de juillet, elle les met presque tous en retard (EDA lab, partie « Les ratios de paiement »). Le correctif applique aux trois premiers mois la règle de juillet-août, puis la définition de la règle 1 est appliquée aux codifications corrigées :
- facture du mois positive (`BILL_AMTn` > 0) et moins de 1,5 % payé le mois suivant (`PAY_AMT(n-1)`) ;
- client codifié -1, 0 ou 1 ce mois-là, et -1 ou 0 le mois précédent (pas après un -2 : report de paiement accepté par la banque) ; en avril, mars n'est pas visible : seule la codification d'avril compte ;
- la codification devient 2 ; avril, mai et juin sont traités dans l'ordre.

Le correctif ne touche pas septembre ni août : il ne peut changer ni la population en incident à M ni le retard imminent, seulement l'historique des clients restés dans le ML.

In [6]:
def uniformiser_retards(d):
    # Codifications 2 d'avril à juin uniformisées sur la règle de juillet-août
    d2 = d.copy()
    for n in (6, 5, 4):   # avril, mai, juin, dans l'ordre
        facture, paiement = d2[f'BILL_AMT{n}'], d2[f'PAY_AMT{n - 1}']
        cible = (facture > 0) & (paiement < 0.015 * facture) & d2[f'PAY_{n}'].isin([-1, 0, 1])
        if n < 6:
            cible &= d2[f'PAY_{n + 1}'].isin([-1, 0])   # client sain (hors -2) le mois précédent
        d2.loc[cible, f'PAY_{n}'] = 2
    return d2

df_train_unif = uniformiser_retards(df_train)
changees = (df_train_unif[pay_cols] != df_train[pay_cols])
print(f"Codifications passées à 2 : {int(changees.values.sum())} | clients touchés : {int(changees.any(axis=1).sum())}")
display(changees[['PAY_6', 'PAY_5', 'PAY_4']].sum().rename({'PAY_6': 'avril', 'PAY_5': 'mai', 'PAY_4': 'juin'}).to_frame('Codifications passées à 2'))

final2b_train = statut_incident(df_train_unif)

print("Comparaison règle 1 / règle 2, par statut à M :")
comparaison = {}
for nom, f in [('Règle 1', final2_train), ('Règle 2', final2b_train)]:
    for s in statuts_r2:
        masque = f['STATUT_INCIDENT'] == s
        comparaison[(nom, 'Clients')] = comparaison.get((nom, 'Clients'), {}) | {s: int(masque.sum())}
        comparaison[(nom, 'Défaut (%)')] = comparaison.get((nom, 'Défaut (%)'), {}) | {s: round(df_train.loc[masque, 'dpnm'].mean() * 100, 1)}
display(pd.DataFrame(comparaison).rename_axis('Statut à M'))

print("Passages d'un statut à l'autre (lignes : règle 1, colonnes : règle 2) :")
display(pd.crosstab(final2_train['STATUT_INCIDENT'].rename('Règle 1'), final2b_train['STATUT_INCIDENT'].rename('Règle 2'), margins=True))

print("Population en incident à M (retirée du ML) :")
display(pd.DataFrame({
    'Règle 1': validation(df_train, final2_train['STATUT_INCIDENT'] == 'En incident'),
    'Règle 2': validation(df_train, final2b_train['STATUT_INCIDENT'] == 'En incident'),
}).T.round(2))

Codifications passées à 2 : 1504 | clients touchés : 1462


,Codifications passées à 2
avril,496
mai,652
juin,356


Comparaison règle 1 / règle 2, par statut à M :


Règle 1            Règle 2           
                    Clients Défaut (%) Clients Défaut (%)
Statut à M                                               
En incident            3019       58.9    3019       58.9
Retard imminent         779       65.1     779       65.1
Sorti de l'incident    1373       30.4    1774       27.6
Retard régularisé      1397       23.6    2023       20.1
Jamais en retard      15193       11.5   14166       11.3

Passages d'un statut à l'autre (lignes : règle 1, colonnes : règle 2) :


Règle 2,En incident,Jamais en retard,Retard imminent,Retard régularisé,Sorti de l'incident,All
Règle 1,,,,,,
En incident,3019,0,0,0,0,3019
Jamais en retard,0,14166,0,738,289,15193
Retard imminent,0,0,779,0,0,779
Retard régularisé,0,0,0,1285,112,1397
Sorti de l'incident,0,0,0,0,1373,1373
All,3019,14166,779,2023,1774,21761


Population en incident à M (retirée du ML) :


,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Règle 1,3019.0,13.87,58.93,37.24,16.0
Règle 2,3019.0,13.87,58.93,37.24,16.0


**Comment lire** : le nombre de codifications passées à 2 par mois ; puis, pour chaque statut, le nombre de clients et leur taux de défaut sous les deux règles ; le tableau croisé montre d'où viennent les clients qui changent de statut ; le dernier tableau compare la population en incident à M.

**Interprétation**
La population en incident à M est identique sous les deux règles, comme le retard imminent : le correctif ne touche que avril, mai et juin.
Il donne en revanche un historique à des clients jamais en retard sous la règle 1 : une partie devient « retard régularisé », une autre « sorti de l'incident ». Ces clients étant moins risqués que les anciens sortis et régularisés, le taux de défaut de ces deux statuts baisse légèrement ; celui des clients jamais en retard bouge peu.
Le choix entre les deux règles porte donc sur l'historique transmis au ML (`FLAG_INCIDENT`, `MOIS_SORTIE_INCIDENT`, `FLAG_RETARD`, `MOIS_SORTIE_RETARD`), pas sur les populations retirées.

## 5. Validation unique sur le test

Exécutée une seule fois, le 10/10/2026, une fois la définition figée et après la correction des codifications 1 : le test n'a servi à ajuster aucune règle. Elle porte sur la **population en incident à M** ; le tableau compare le train et le test.

**Découpage corrigé le 11/10/2026** : le périmètre S12 est désormais filtré **avant** le découpage, comme le veut la règle du projet (le découpage portait jusque-là sur tout `cleaned2`, le filtre venant après). Le test a donc changé de composition : la validation porte sur ce nouveau test, jamais lu auparavant, et la définition n'a pas été modifiée.

Le **retard imminent** a été retiré du ML ensuite (décision du 11/10/2026), pour une raison métier (section 4.1) et non sur ses chiffres : il est mesuré sur le train seulement, et le test n'est pas relu pour lui.

In [7]:
final2_test = statut_incident(df_test)
display(pd.DataFrame({
    'Train': validation(df_train, final2_train['STATUT_INCIDENT'] == 'En incident'),
    'Test': validation(df_test, final2_test['STATUT_INCIDENT'] == 'En incident'),
}).T.round(2).rename_axis('Population en incident à M'))

,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Population en incident à M,,,,,
Train,3019.0,13.87,58.93,37.24,16.00
Test,735.0,13.51,60.27,37.07,15.98


## 6. Indicateurs à reporter dans les notebooks ML
Définition retenue : deux codifications de retard successives, ou une codification de 3 ou plus (section 4), fonction `statut_incident`.
- les clients **en incident à M** (`FLAG_INCIDENT` = 1 et `MOIS_SORTIE_INCIDENT` = -1) sont retirés du dataset ML et prédits en défaut par la règle ;
- les clients en **retard imminent** (`FLAG_RETARD_IMMINENT` = 1) sont retirés aussi, comme une population à part, et prédits en défaut ;
- les autres clients (`Sorti de l'incident`, `Retard régularisé`, `Jamais en retard`) restent dans le ML avec `FLAG_INCIDENT`, `MOIS_SORTIE_INCIDENT`, `FLAG_RETARD` et `MOIS_SORTIE_RETARD`.

In [8]:
final2 = statut_incident(df)
print("Statut à M sur le périmètre S12 (définition 2) :")
display(final2['STATUT_INCIDENT'].value_counts().reindex(statuts_r2).rename_axis('Statut à M').to_frame('Nb clients'))

Statut à M sur le périmètre S12 (définition 2) :


,Nb clients
Statut à M,
En incident,3754
Retard imminent,989
Sorti de l'incident,1706
Retard régularisé,1716
Jamais en retard,19037


## 7. Mise à l'épreuve sur le dataset initial (30 000 lignes)

La définition retenue (section 6) est appliquée au **dataset initial complet** `creditcard_pret_ingestion.csv` : 30 000 lignes, **aucun filtre**, codes PAY_n bruts (seul le nettoyage structurel de `EDUCATION` et `MARRIAGE` est appliqué).

On se compare au « monde réel » du dataset : que se passe-t-il si l'on retire les deux populations du dataset de base ?
- quelle part des clients et des défauts part avec chacune,
- quel taux de défaut reste dans le dataset sans elles.

*Ce n'est pas un test indépendant : le dataset initial contient le train sur lequel la règle a été construite.*

In [9]:
df_initial = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv")
final_initial = statut_incident(df_initial)
retires_initial = retires_du_ml(final_initial)

def bilan(d):
    return pd.Series({
        'Nb clients': len(d),
        'Part des clients (%)': len(d) / len(df_initial) * 100,
        'Nb défauts': int(d['dpnm'].sum()),
        'Part des défauts (%)': d['dpnm'].sum() / df_initial['dpnm'].sum() * 100,
        'Taux défaut (%)': d['dpnm'].mean() * 100,
    })

print("Retrait des deux populations (en incident à M ; retard imminent) :")
display(pd.DataFrame({
    'Dataset initial': bilan(df_initial),
    'En incident à M, retiré': bilan(df_initial[retires_initial['En incident à M']]),
    'Retard imminent, retiré': bilan(df_initial[retires_initial['Retard imminent']]),
    'Dataset sans les deux': bilan(df_initial[~retires_initial.any(axis=1)]),
}).T.round(2))

Retrait des deux populations (en incident à M ; retard imminent) :


,Nb clients,Part des clients (%),Nb défauts,Part des défauts (%),Taux défaut (%)
Dataset initial,30000.0,100.00,6636.0,100.00,22.12
"En incident à M, retiré",2139.0,7.13,1526.0,23.00,71.34
"Retard imminent, retiré",991.0,3.30,651.0,9.81,65.69
Dataset sans les deux,26870.0,89.57,4459.0,67.19,16.59


In [10]:
print("Statuts à M sur le dataset initial :")
display(pd.DataFrame({s: bilan(df_initial[final_initial['STATUT_INCIDENT'] == s]) for s in statuts_r2}).T.round(2)
        .rename_axis('Statut à M'))

Statuts à M sur le dataset initial :


,Nb clients,Part des clients (%),Nb défauts,Part des défauts (%),Taux défaut (%)
Statut à M,,,,,
En incident,2139.0,7.13,1526.0,23.00,71.34
Retard imminent,991.0,3.30,651.0,9.81,65.69
Sorti de l'incident,2848.0,9.49,1080.0,16.27,37.92
Retard régularisé,2402.0,8.01,623.0,9.39,25.94
Jamais en retard,21620.0,72.07,2756.0,41.53,12.75


**Interprétation**
- Avec les codifications d'origine, les 1 de septembre ne sont pas lus comme des retards : une partie des clients en retard en août n'y apparaît ni en incident ni en retard imminent. Les deux populations y sont donc plus petites que sur les données nettoyées.
- Retirer les deux populations enlève une petite part des clients mais une part bien plus grande des défauts : leur taux de défaut reste très élevé sur l'ensemble du dataset, du même ordre que sur le train.
- Le dataset sans elles a un taux de défaut nettement plus bas que le dataset initial : c'est la population que le ML aura à traiter, plus homogène mais avec moins de signal.
- La gradation du risque entre les statuts (en incident, retard imminent, sorti de l'incident, retard régularisé, jamais en retard) est la même que sur le train.
- Contrepartie à assumer : une partie des clients retirés ne fera pas défaut mais sera prédite en défaut par la règle.

### 7.1 Métriques des règles de retrait
Chaque règle est évaluée **uniquement sur la population qu'elle traite**, tous les clients étant prédits en défaut : les clients en incident à M (`FLAG_INCIDENT` = 1 et `MOIS_SORTIE_INCIDENT` = -1), puis les clients en retard imminent (`FLAG_RETARD_IMMINENT` = 1). Les autres clients relèvent du ML et ne sont pas comptés ici.
*Sur ce périmètre, le recall vaut 100 % par construction : toutes les erreurs d'une règle sont des faux positifs, et la précision est la métrique qui compte. La part des défauts du dataset captés par la règle est donnée pour situer son poids. Le test n'est pas lu ici (section 5).*

In [11]:
from sklearn.metrics import fbeta_score, recall_score, precision_score

def metriques_regle(d, population):
    retire = retires_du_ml(statut_incident(d))[population]
    y = d.loc[retire, 'dpnm']
    y_pred = pd.Series(1, index=y.index)          # tous les clients retirés sont prédits en défaut
    return pd.Series({
        'Clients retirés': retire.sum(),
        'Vrais positifs': int(y.sum()),
        'Faux positifs': int((y == 0).sum()),
        'Précision (%)': precision_score(y, y_pred) * 100,
        'Recall (%)': recall_score(y, y_pred) * 100,
        'F2': fbeta_score(y, y_pred, beta=2),
        'Part des défauts du dataset captés (%)': y.sum() / d['dpnm'].sum() * 100,
    })

display(pd.DataFrame({
    (jeu, population): metriques_regle(d, population)
    for jeu, d in [('Train (S12)', df_train), ('Dataset initial (30 000)', df_initial)]
    for population in ['En incident à M', 'Retard imminent']
}).T.round(4))

Clients retirés  Vrais positifs  \
Train (S12)              En incident à M           3019.0          1779.0   
                         Retard imminent            779.0           507.0   
Dataset initial (30 000) En incident à M           2139.0          1526.0   
                         Retard imminent            991.0           651.0   

                                          Faux positifs  Précision (%)  \
Train (S12)              En incident à M         1240.0        58.9268   
                         Retard imminent          272.0        65.0834   
Dataset initial (30 000) En incident à M          613.0        71.3417   
                         Retard imminent          340.0        65.6912   

                                          Recall (%)      F2  \
Train (S12)              En incident à M       100.0  0.8777   
                         Retard imminent       100.0  0.9031   
Dataset initial (30 000) En incident à M       100.0  0.9256   
                         Retard imminent       100.0  0.9054   

                                          Part des défauts du dataset captés (%)  
Train (S12)              En incident à M                                 37.2409  
                         Retard imminent                                 10.6134  
Dataset initial (30 000) En incident à M                                 22.9958  
                         Retard imminent                                  9.8101

**Interprétation**
Sur la population qu'elle traite, chaque règle se trompe peu : la grande majorité des clients retirés font défaut, et ses seules erreurs sont des clients sains prédits en défaut (faux positifs).
Les deux règles ne traitent qu'une partie des défauts du dataset : les autres seront détectés par le ML sur la population restante. La performance finale se mesurera sur le **système complet** (les deux règles + ML).

---
## Annexe 1 : les codifications de retard (PAY_n >= 2)

Étude reprise de l'EDA lab, qui a fait naître la sous-population contentieuse, menée ici sur le **train**. Les codifications sont lues telles quelles ; les montants suivent la lecture de l'audit (`PAY_AMTn` règle `BILL_AMT(n+1)`).

**Codification figée à 2 sur les 6 mois**

In [12]:
import pandas as pd

# 1. Identification des colonnes de statut de paiement (PAY_1 à PAY_6)
cols_pay = [
    c
    for c in ['PAY_1', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    if c in df_train.columns
]

# 2. Masque pour interroger la présence continue de la valeur 2
mask_always_2 = (df_train[cols_pay] == 2).all(axis=1)

# 3. Décompte total dans la base
count_always_2 = mask_always_2.sum()

print(
    f"Nombre total de clients avec PAY_n == 2 sur les 6 mois : {count_always_2}"
)

# 4. Isolement du sous-ensemble de clients
df_always_2 = df_train[mask_always_2]

# Affichage des colonnes clés
cols_display = [
    c
    for c in ['ID', 'LIMIT_BAL']
    + cols_pay
    + ['BILL_AMT1', 'BILL_AMT6', 'PAY_AMT1', 'dpnm']
    if c in df_train.columns
]
display(df_always_2[['ID', 'LIMIT_BAL'] + cols_pay + ['BILL_AMT1', 'BILL_AMT6', 'PAY_AMT1', 'dpnm']].head(10))
print(f"taux de défaut sur cette population : {round(df_always_2['dpnm'].sum()/len(df_always_2)*100,2)}%")

Nombre total de clients avec PAY_n == 2 sur les 6 mois : 657


,ID,LIMIT_BAL,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT6,PAY_AMT1,dpnm
14396,14981,30000,2,2,2,2,2,2,28224,29793,1800,1
7594,7926,50000,2,2,2,2,2,2,32899,36588,2000,1
16337,16992,50000,2,2,2,2,2,2,36665,14974,4626,1
23342,24257,280000,2,2,2,2,2,2,106907,112607,6900,0
6190,6456,80000,2,2,2,2,2,2,53541,58351,2600,0
995,1045,20000,2,2,2,2,2,2,16233,19994,5383,1
18752,19490,180000,2,2,2,2,2,2,130304,149133,13600,1
22810,23705,50000,2,2,2,2,2,2,40926,44838,3000,1
9708,10125,20000,2,2,2,2,2,2,6313,9081,2426,0
11893,12393,30000,2,2,2,2,2,2,28073,28181,1800,1


taux de défaut sur cette population : 70.02%


**Interprétation**
Une codification gelée à 2 sur les six mois concerne quelques centaines de clients, avec un taux de défaut très élevé : cela ressemble à une gestion contentieuse. Ces comptes sont-ils gelés ?

**La dette des clients figés à 2 : équation de gel et mois sans paiement**  
Si un compte est gelé, la facture suivante vaut la précédente moins le paiement (`BILL_AMT(n+1)` - `PAY_AMTn` = `BILL_AMTn`), et la dette ne grossit que par les intérêts et les frais quand rien n'est payé.

In [13]:
import pandas as pd

# Paires de transition : BILL_AMT_{n+1} - PAY_AMT_n doit égaler BILL_AMT_n
transitions = [
    ('M2 -> M1', 'BILL_AMT2', 'PAY_AMT1', 'BILL_AMT1'),
    ('M3 -> M2', 'BILL_AMT3', 'PAY_AMT2', 'BILL_AMT2'),
    ('M4 -> M3', 'BILL_AMT4', 'PAY_AMT3', 'BILL_AMT3'),
    ('M5 -> M4', 'BILL_AMT5', 'PAY_AMT4', 'BILL_AMT4'),
    ('M6 -> M5', 'BILL_AMT6', 'PAY_AMT5', 'BILL_AMT5'),
]

# Calcul des écarts pour chaque transition
df_diffs = pd.DataFrame(index=df_always_2.index)

for label, bill_next, pay_curr, bill_curr in transitions:
    # Écart = (Encours M+1 - Versement M) - Encours M
    df_diffs[label] = (
        df_always_2[bill_next] - df_always_2[pay_curr]
    ) - df_always_2[bill_curr]

# 1. Égalité stricte (Écart == 0)
exact_match_per_trans = (df_diffs == 0).sum()
exact_match_all_5 = (df_diffs == 0).all(axis=1).sum()

# 2. Égalité avec tolérance de 100 NT$ (pour prendre en compte de légers agios résiduels)
tol_match_per_trans = (df_diffs.abs() <= 100).sum()
tol_match_all_5 = (df_diffs.abs() <= 100).all(axis=1).sum()

total = len(df_always_2)

print(f'=== Vérification de l\'équation de gel sur les {total} clients ===\n')

print('--- Égalité stricte (0 NT$ d\'écart) ---')
for label in df_diffs.columns:
    count = exact_match_per_trans[label]
    print(f'{label} : {count} clients ({count / total * 100:.2f}%)')
print(
    f'\nParfaitement vérifié sur les 5 transitions : {exact_match_all_5} ({exact_match_all_5 / total * 100:.2f}%)\n'
)

print('--- Avec tolérance d\'agios (écart <= 100 NT$) ---')
for label in df_diffs.columns:
    count = tol_match_per_trans[label]
    print(f'{label} : {count} clients ({count / total * 100:.2f}%)')
print(
    f'\nVérifié sur les 5 transitions (avec tolérance) : {tol_match_all_5} ({tol_match_all_5 / total * 100:.2f}%)'
)

=== Vérification de l'équation de gel sur les 657 clients ===

--- Égalité stricte (0 NT$ d'écart) ---
M2 -> M1 : 14 clients (2.13%)
M3 -> M2 : 14 clients (2.13%)
M4 -> M3 : 15 clients (2.28%)
M5 -> M4 : 15 clients (2.28%)
M6 -> M5 : 15 clients (2.28%)

Parfaitement vérifié sur les 5 transitions : 13 (1.98%)

--- Avec tolérance d'agios (écart <= 100 NT$) ---
M2 -> M1 : 15 clients (2.28%)
M3 -> M2 : 16 clients (2.44%)
M4 -> M3 : 20 clients (3.04%)
M5 -> M4 : 17 clients (2.59%)
M6 -> M5 : 17 clients (2.59%)

Vérifié sur les 5 transitions (avec tolérance) : 13 (1.98%)


In [14]:
import pandas as pd

# Transitions M+1 -> M
transitions = [
    ('BILL_AMT2', 'BILL_AMT1', 'PAY_AMT1'),
    ('BILL_AMT3', 'BILL_AMT2', 'PAY_AMT2'),
    ('BILL_AMT4', 'BILL_AMT3', 'PAY_AMT3'),
    ('BILL_AMT5', 'BILL_AMT4', 'PAY_AMT4'),
    ('BILL_AMT6', 'BILL_AMT5', 'PAY_AMT5'),
]

growths = []

for bill_prev, bill_curr, pay_curr in transitions:
    # Isolement des mois sans aucun remboursement
    mask_no_pay = df_always_2[pay_curr] == 0
    # Variation de la dette : BILL_AMT_n - BILL_AMT_{n+1}
    diff = (
        df_always_2.loc[mask_no_pay, bill_curr]
        - df_always_2.loc[mask_no_pay, bill_prev]
    )
    growths.append(diff)

all_growths = pd.concat(growths)

print(
    "=== Variation mensuelle de l'encours sans paiement (df_always_2) ==="
)
print(f"Nombre de mois observés sans paiement : {len(all_growths)}")
print(
    f"Mois avec augmentation de la dette : {(all_growths > 0).mean() * 100:.2f}%"
)
print(
    f"Mois avec dette strictement identique (0 NT$)  : {(all_growths == 0).mean() * 100:.2f}%"
)
print(f"Augmentation médiane mensuelle             : {all_growths.median():.0f} NT$")
print(f"Augmentation moyenne mensuelle             : {all_growths.mean():.0f} NT$")

=== Variation mensuelle de l'encours sans paiement (df_always_2) ===
Nombre de mois observés sans paiement : 865
Mois avec augmentation de la dette : 91.68%
Mois avec dette strictement identique (0 NT$)  : 8.21%
Augmentation médiane mensuelle             : 781 NT$
Augmentation moyenne mensuelle             : 2272 NT$


In [15]:
import pandas as pd

# 1. Sélection dynamique des colonnes
pay_status_cols = [
    c for c in df_train.columns if c.startswith('PAY_') and not c.startswith('PAY_AMT')
]
pay_amt_cols = sorted([c for c in df_train.columns if c.startswith('PAY_AMT')])

# 2. Filtrage de la population "PAY_n == 2 tout le temps"
mask_pay2_6m = (df_train[pay_status_cols] == 2).all(axis=1)
df_chronique = df_train[mask_pay2_6m].copy()

# 3. Compte du nombre de mois avec PAY_AMT == 0 par client (0 à 6)
zeros_par_client = (df_chronique[pay_amt_cols] == 0).sum(axis=1)

# 4. Distribution et impact sur le taux de défaut (dpnm)
distrib = (
    zeros_par_client.value_counts()
    .sort_index()
    .reset_index()
    .rename(
        columns={
            'index': 'Mois avec PAY_AMT = 0',
            0: 'Mois avec PAY_AMT = 0',
            'count': 'Nb_Clients',
        }
    )
)

distrib['Pct_Population_incident_%'] = (
    distrib['Nb_Clients'] / len(df_chronique) * 100
).round(2)

if 'dpnm' in df_chronique.columns:
    taux_dpnm = (
        df_chronique.groupby(zeros_par_client)['dpnm'].mean() * 100
    ).round(2)
    distrib['Taux_Defaut_dpnm_%'] = distrib['Mois avec PAY_AMT = 0'].map(
        taux_dpnm
    )

display(distrib)

# 5. Métriques globales
total_cellules_amt = len(df_chronique) * len(pay_amt_cols)
total_zeros = (df_chronique[pay_amt_cols] == 0).sum().sum()

print(f'=== BILAN DES PAIEMENT NULS (POPULATION EN RETARD DEPUIS 6 MOIS) ===')
print(f'Nombre de clients concernés : {len(df_chronique):,}'.replace(',', ' '))
print(
    f'Total d\'échéances avec PAY_AMT == 0 : {total_zeros:,} sur {total_cellules_amt:,} ({total_zeros/total_cellules_amt*100:.2f} %)'.replace(
        ',', ' '
    )
)
print(
    f'Clients avec ZERO PAIEMENT ABSOLU sur 6 mois (6/6) : {(zeros_par_client == 6).sum()} ({(zeros_par_client == 6).mean()*100:.2f} %)'
)

,Mois avec PAY_AMT = 0,Nb_Clients,Pct_Population_incident_%,Taux_Defaut_dpnm_%
0,0,107,25.24,79.44
1,1,129,30.42,78.29
2,2,128,30.19,77.34
3,3,46,10.85,80.43
4,4,1,0.24,0.00
5,5,1,0.24,100.00
6,6,12,2.83,100.00


=== BILAN DES PAIEMENT NULS (POPULATION EN RETARD DEPUIS 6 MOIS) ===
Nombre de clients concernés : 424
Total d'échéances avec PAY_AMT == 0 : 604 sur 2 544 (23.74 %)
Clients avec ZERO PAIEMENT ABSOLU sur 6 mois (6/6) : 12 (2.83 %)


**Retards de 3 et plus**

In [16]:
import pandas as pd

# 1. Sélection des colonnes de statut (PAY_1 à PAY_6)
pay_status_cols = [f'PAY_{i}' for i in range(1, 7)]

# 2. Condition : Au moins un mois avec PAY_n > 2
mask_pay_gt_2 = (df_train[pay_status_cols] > 2).any(axis=1)

# 3. Calcul des métriques
df_gt_2 = df_train[mask_pay_gt_2]
nb_clients = len(df_gt_2)
pct_portefeuille = (nb_clients / len(df_train)) * 100
taux_defaut = df_gt_2['dpnm'].mean() * 100 if 'dpnm' in df_train.columns else None

# 4. Synthèse des résultats
res_gt2 = pd.DataFrame([
    {
        'Périmètre': 'Au moins un PAY_n > 2 sur 6 mois',
        'Nb Clients': f'{nb_clients:,}'.replace(',', ' '),
        'Part Portefeuille (%)': f'{pct_portefeuille:.2f} %',
        'Taux de Défaut dpnm (%)': (
            f'{taux_defaut:.2f} %' if taux_defaut is not None else 'N/A'
        ),
    }
])

display(res_gt2)

# 5. Ventilation détaillée par statut maximum atteint
df_train['Max_PAY_6m'] = df_train[pay_status_cols].max(axis=1)
ventilation = (
    df_train.groupby('Max_PAY_6m')
    .agg(
        Nb_Clients=('dpnm', 'count'),
        Pct_Portefeuille=('dpnm', lambda x: len(x) / len(df_train) * 100),
        Taux_Defaut_dpnm=('dpnm', lambda x: x.mean() * 100),
    )
    .reset_index()
)

display(ventilation[ventilation['Max_PAY_6m'] > 2])

,Périmètre,Nb Clients,Part Portefeuille (%),Taux de Défaut dpnm (%)
0,Au moins un PAY_n > 2 sur 6 mois,949,4.36 %,62.49 %


,Max_PAY_6m,Nb_Clients,Pct_Portefeuille,Taux_Defaut_dpnm
4,3,619,2.844538,61.550889
5,4,176,0.808786,64.204545
6,5,57,0.261936,52.631579
7,6,23,0.105694,56.521739
8,7,53,0.243555,86.792453
9,8,21,0.096503,47.619048


In [17]:
import pandas as pd

# 1. Sélection des colonnes de statut (PAY_1 à PAY_6)
pay_status_cols = [f'PAY_{i}' for i in range(1, 7)]

# 2. Condition : PAY_n > 2 TOUS LES MOIS sur les 6 mois
mask_gt2_strict_6m = (df_train[pay_status_cols] > 2).all(axis=1)

# 3. Métriques pour ce segment
df_gt2_6m = df_train[mask_gt2_strict_6m]
nb_clients = len(df_gt2_6m)
pct_pop = (nb_clients / len(df_train)) * 100
dpnm_moyen = (
    df_gt2_6m['dpnm'].mean() * 100 if 'dpnm' in df_train.columns and nb_clients > 0 else 0
)

# 4. Affichage du résultat
print(f'=== SEGMENT STRICT : PAY_n > 2 SUR LES 6 MOIS ===')
print(f'Nb clients : {nb_clients:,}'.replace(',', ' '))
print(f'Part portefeuille : {pct_pop:.2f} %')
print(f'Taux de défaut (dpnm) : {dpnm_moyen:.2f} %')

# 5. Détail par profil exact pour comprendre la structure interne
if nb_clients > 0:
  profils = (
      df_gt2_6m.groupby(pay_status_cols)
      .agg(
          Nb_Clients=('dpnm', 'count'),
          Taux_Defaut_dpnm=('dpnm', lambda x: x.mean() * 100),
      )
      .reset_index()
  )
  display(profils)

=== SEGMENT STRICT : PAY_n > 2 SUR LES 6 MOIS ===
Nb clients : 19
Part portefeuille : 0.09 %
Taux de défaut (dpnm) : 52.63 %


,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,Nb_Clients,Taux_Defaut_dpnm
0,3,3,4,4,4,4,1,0.000000
1,3,3,4,4,5,6,1,100.000000
2,4,3,4,4,4,3,1,100.000000
3,4,4,4,4,4,4,1,100.000000
4,8,7,6,5,4,3,15,46.666667


**Séries de 2 depuis septembre et durée du blocage à 2** (provisoire : partent de `PAY_1`)

In [18]:
import pandas as pd

# 1. Alignement des colonnes de statut ordonnées (de M-1 à M-6)
pay_m1 = 'PAY_0' if 'PAY_0' in df_train.columns else 'PAY_1'
start_idx = 0 if pay_m1 == 'PAY_0' else 1

pay_status_cols = [
    f'PAY_{i}'
    for i in range(start_idx, start_idx + 6)
    if f'PAY_{i}' in df_train.columns
]

results = []

# 2. Calcul du taux de défaut pour chaque fenêtre consécutive n (de 1 à 6 mois)
for n in range(1, len(pay_status_cols) + 1):
    cols_window = pay_status_cols[:n]

    # Condition : PAY == 2 consécutif de M-1 jusqu'à M-n
    mask_consecutive = (df_train[cols_window] == 2).all(axis=1)

    nb_clients = mask_consecutive.sum()
    pct_pop = (nb_clients / len(df_train)) * 100
    dpnm_moyen = (
        df_train.loc[mask_consecutive, 'dpnm'].mean() * 100 if nb_clients > 0 else 0
    )

    results.append({
        'Série consécutive (depuis M-1)': f'Au moins {n} mois (M-1 à M-{n})',
        'Nb clients': f'{nb_clients:,}'.replace(',', ' '),
        'Part portefeuille (%)': round(pct_pop, 2),
        'Taux de défaut dpnm (%)': round(dpnm_moyen, 2),
    })

# 3. Affichage
df_res = pd.DataFrame(results)
display(df_res)

,Série consécutive (depuis M-1),Nb clients,Part portefeuille (%),Taux de défaut dpnm (%)
0,Au moins 1 mois (M-1 à M-1),3 428,15.75,58.98
1,Au moins 2 mois (M-1 à M-2),2 468,11.34,57.41
2,Au moins 3 mois (M-1 à M-3),1 555,7.15,63.15
3,Au moins 4 mois (M-1 à M-4),1 114,5.12,65.35
4,Au moins 5 mois (M-1 à M-5),819,3.76,68.38
5,Au moins 6 mois (M-1 à M-6),657,3.02,70.02


In [19]:
import pandas as pd

pay_status_cols = [f'PAY_{i}' for i in range(1, 7)]


# Fonction pour calculer la durée exacte du blocage continu à 2 depuis PAY_1
def duree_exacte_pay2(row):
  count = 0
  for col in pay_status_cols:
    if row[col] == 2:
      count += 1
    else:
      break
  return count


# 1. Calcul de la durée exacte (0 à 6 mois)
df_train['Duree_PAY2_Consecutif'] = df.apply(duree_exacte_pay2, axis=1)

# 2. Mapping des tranches disjointes
labels = {
    1: '1. Exactement 1 mois (PAY_1=2, PAY_2!=2)',
    2: '2. Exactement 2 mois (PAY_1..2=2, PAY_3!=2)',
    3: '3. Exactement 3 mois (PAY_1..3=2, PAY_4!=2)',
    4: '4. Exactement 4 mois (PAY_1..4=2, PAY_5!=2)',
    5: '5. Exactement 5 mois (PAY_1..5=2, PAY_6!=2)',
    6: '6. 6 mois complets (PAY_1..6=2)',
}

df_train['Segment_PAY2_Disjoint'] = df_train['Duree_PAY2_Consecutif'].map(labels)

# 3. Tableau de ventilation étanche
analyse_disjointe = (
    df_train[df_train['Duree_PAY2_Consecutif'] > 0]
    .groupby('Segment_PAY2_Disjoint')
    .agg(
        Nb_Clients=('dpnm', 'count'),
        Pct_Portefeuille=('dpnm', lambda x: len(x) / len(df_train) * 100),
        Taux_Defaut_dpnm=('dpnm', lambda x: x.mean() * 100),
    )
    .reset_index()
)

display(analyse_disjointe)

,Segment_PAY2_Disjoint,Nb_Clients,Pct_Portefeuille,Taux_Defaut_dpnm
0,"1. Exactement 1 mois (PAY_1=2, PAY_2!=2)",960,4.411562,63.020833
1,"2. Exactement 2 mois (PAY_1..2=2, PAY_3!=2)",913,4.195579,47.645126
2,"3. Exactement 3 mois (PAY_1..3=2, PAY_4!=2)",441,2.026561,57.596372
3,"4. Exactement 4 mois (PAY_1..4=2, PAY_5!=2)",295,1.355636,56.949153
4,"5. Exactement 5 mois (PAY_1..5=2, PAY_6!=2)",162,0.744451,61.728395
5,6. 6 mois complets (PAY_1..6=2),657,3.019163,70.015221


**Règle simple : un seul retard (PAY_n >= 2) suffit**

In [20]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Sélection des colonnes de statut (PAY_1 à PAY_6)
pay_status_cols = [f'PAY_{i}' for i in range(1, 7)]

# 2. Définition de la prédiction par la Hard Rule (PAY_n >= 2 sur au moins 1 mois)
y_pred_hard_rule = (df_train[pay_status_cols] >= 2).any(axis=1).astype(int)
y_true = df_train['dpnm']

# 3. Calcul de la matrice de confusion
# Matrice [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_true, y_pred_hard_rule)
tn, fp, fn, tp = cm.ravel()

# 4. Formattage sous forme de DataFrame
df_cm = pd.DataFrame(
    cm,
    index=['Réel : Non-Défaut (0)', 'Réel : Défaut (1)'],
    columns=['Predit : Non-Défaut (0)', 'Predit : Défaut (1)'],
)

display(df_cm)

# 5. Métriques clés
print('=== BILAN PERFORMANCES HARD RULE (PAY_n >= 2) ===')
print(f'Vrais Positifs (TP)  : {tp:,}'.replace(',', ' '))
print(f'Faux Positifs (FP)   : {fp:,} (Sains classés à tort en incident)'.replace(',', ' '))
print(f'Vrais Négatifs (TN)  : {tn:,}'.replace(',', ' '))
print(f'Faux Négatifs (FN)   : {fn:,} (Défauts manqués par la règle)'.replace(',', ' '))
print('-' * 50)
print(f'Rappel (Sensitivity) : {tp / (tp + fn) * 100:.2f} % (Taux de captation du défaut)')
print(f'Précision            : {tp / (tp + fp) * 100:.2f} % (Taux de vrais défauts dans la population en incident)')
print(f'Spécificité          : {tn / (tn + fp) * 100:.2f} % (Taux de protection des clients sains)')
print(f'Taux d\'erreur global : {(fp + fn) / len(df_train) * 100:.2f} %')

,Predit : Non-Défaut (0),Predit : Défaut (1)
Réel : Non-Défaut (0),13449,3535
Réel : Défaut (1),1744,3033


=== BILAN PERFORMANCES HARD RULE (PAY_n >= 2) ===
Vrais Positifs (TP)  : 3 033
Faux Positifs (FP)   : 3 535 (Sains classés à tort en incident)
Vrais Négatifs (TN)  : 13 449
Faux Négatifs (FN)   : 1 744 (Défauts manqués par la règle)
--------------------------------------------------
Rappel (Sensitivity) : 63.49 % (Taux de captation du défaut)
Précision            : 46.18 % (Taux de vrais défauts dans la population en incident)
Spécificité          : 79.19 % (Taux de protection des clients sains)
Taux d'erreur global : 24.26 %


**Interprétation**
Le dataset ne dit pas si ces comptes sont gelés, au contentieux ou dans une autre gestion. Une codification de retard qui dure a pourtant une signification particulière, avec un risque très élevé : elle justifie de traiter à part les clients en retard installé (deux codifications de retard successives), plutôt que tout client ayant eu un seul retard.

## Annexe 2 : règles exploratoires
Règles testées avant la définition retenue (section 4). Elles sont conservées pour tracer le raisonnement. Le « faux rétrogradage » (R2) et les règles qui l'incluaient (R3, R7) reposaient sur la lecture décalée des colonnes et ont été retirés ; R8 est recalculée sans R2.

### A.1 Règles candidates

| Règle | Définition | Logique métier |
|---|---|---|
| `R1_chronique` | PAY_n = 2 sur les 6 mois | codification figée, gestion hors circuit normal |
| `R4_retard_6m` | PAY_n >= 2 sur les 6 mois | retard sur toute la période observée (inclut R1) |
| `R5_retard_profond` | au moins un PAY_n > 2 sur les 6 mois | un seul retard d'au moins 3 mois suffit |
| `R6_retard_continu` | PAY_n >= 2 sans interruption de M-1 jusqu'à M-k, avec 2 <= k <= 5 | retard installé depuis au moins 2 mois et toujours en cours à M-1 (6 mois = R4) |
| `R8_elargie` | R1 ou R4 ou R5 ou R6 | toutes les situations de retard profond, figé ou installé |

**Règles bis (`_bis`)** : PAY_1 = 1, PAY_2 >= 2 et même règle appliquée sur la fenêtre PAY_2 à PAY_6. Les règles R1, R4 et R6 exigent un retard à M-1 et excluent donc mécaniquement les PAY_1 = 1, alors qu'un passage de >= 2 à 1 en septembre ne signifie pas forcément une sortie. Pour `R6_retard_continu_bis`, la série continue part de M-2 (2 à 4 mois, 5 mois = `R4_retard_6m_bis`).

**Règles ter (`_ter`)** : même règle appliquée sur la fenêtre PAY_2 à PAY_6, **sans regarder PAY_1**, dont la fiabilité est douteuse (codification 1 quasi exclusive à ce mois). Si une règle ter a un taux de défaut proche de sa version de base, PAY_1 n'apporte rien pour détecter l'incident de paiement. La ventilation des clients ter selon leur PAY_1 (A.3) montre ce que ce mois apporte réellement.

In [21]:
# Les règles s'appliquent sur une fenêtre de mois :
# - règles de base : PAY_1 à PAY_6
# - règles bis : PAY_1 == 1, PAY_2 >= 2 et même règle appliquée sur PAY_2 à PAY_6
# - règles ter : même règle appliquée sur PAY_2 à PAY_6, PAY_1 ignoré (fiabilité douteuse)
mois_base = [1, 2, 3, 4, 5, 6]
mois_bis = [2, 3, 4, 5, 6]

def cols(mois):
    return [f'PAY_{n}' for n in mois]

def regle_R1(d, mois):
    return (d[cols(mois)] == 2).all(axis=1)

def regle_R4(d, mois):
    return (d[cols(mois)] >= 2).all(axis=1)

def regle_R5(d, mois):
    return (d[cols(mois)] > 2).any(axis=1)

def regle_R6(d, mois):
    # Durée de la série continue de PAY_n >= 2 en partant du mois le plus récent de la fenêtre
    duree_retard_continu = (d[cols(mois)] >= 2).cumprod(axis=1).sum(axis=1)
    # Au moins 2 mois, fenêtre complète exclue (= R4)
    return duree_retard_continu.between(2, len(mois) - 1)

regles_base = {
    'R1_chronique': regle_R1,
    'R4_retard_6m': regle_R4,
    'R5_retard_profond': regle_R5,
    'R6_retard_continu': regle_R6,
    'R8_elargie': lambda d, mois: regle_R1(d, mois) | regle_R4(d, mois) | regle_R5(d, mois) | regle_R6(d, mois),
}

def condition_bis(d):
    return (d['PAY_1'] == 1) & (d['PAY_2'] >= 2)

regles = {}
for nom, f in regles_base.items():
    regles[nom] = lambda d, f=f: f(d, mois_base)
for nom, f in regles_base.items():
    regles[f'{nom}_bis'] = lambda d, f=f: condition_bis(d) & f(d, mois_bis)
for nom, f in regles_base.items():
    regles[f'{nom}_ter'] = lambda d, f=f: f(d, mois_bis)

masques_train = pd.DataFrame({nom: f(df_train) for nom, f in regles.items()})

### A.2 Description des règles (sans la cible)

In [22]:
volumes = pd.DataFrame({
    'Nb clients': masques_train.sum(),
    'Part train (%)': (masques_train.mean() * 100).round(2),
}).rename_axis('Règle')
display(volumes)

# Recouvrement : part des clients de la règle en ligne également captés par la règle en colonne
recouvrement = pd.DataFrame(
    {c: [(masques_train[l] & masques_train[c]).sum() / max(masques_train[l].sum(), 1) * 100 for l in regles] for c in regles},
    index=list(regles),
).round(1).rename_axis('Règle (ligne) captée par ->')
print("Recouvrement (% des clients de la ligne captés par la colonne) :")
display(recouvrement)

,Nb clients,Part train (%)
Règle,,
R1_chronique,657,3.02
R4_retard_6m,1095,5.03
R5_retard_profond,949,4.36
R6_retard_continu,1924,8.84
R8_elargie,3209,14.75
R1_chronique_bis,0,0.00
R4_retard_6m_bis,0,0.00
R5_retard_profond_bis,0,0.00
R6_retard_continu_bis,0,0.00


Recouvrement (% des clients de la ligne captés par la colonne) :


,R1_chronique,R4_retard_6m,R5_retard_profond,R6_retard_continu,R8_elargie,R1_chronique_bis,R4_retard_6m_bis,R5_retard_profond_bis,R6_retard_continu_bis,R8_elargie_bis,R1_chronique_ter,R4_retard_6m_ter,R5_retard_profond_ter,R6_retard_continu_ter,R8_elargie_ter
Règle (ligne) captée par ->,,,,,,,,,,,,,,,
R1_chronique,100.0,100.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,100.0,100.0,0.0,0.0,100.0
R4_retard_6m,60.0,100.0,40.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,63.6,100.0,36.4,0.0,100.0
R5_retard_profond,0.0,46.2,100.0,33.8,100.0,0.0,0.0,0.0,0.0,0.0,4.1,47.3,84.1,31.8,95.0
R6_retard_continu,0.0,0.0,16.7,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.9,55.5,56.3
R8_elargie,20.5,34.1,29.6,60.0,100.0,0.0,0.0,0.0,0.0,0.0,21.7,34.5,24.9,34.7,73.8
R1_chronique_bis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R4_retard_6m_bis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R5_retard_profond_bis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
R6_retard_continu_bis,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [23]:
profils = {nom: profil_comportement(df_train[masques_train[nom]]) for nom in regles}
tous_ctx = masques_train.any(axis=1)
profils['Hors toutes règles'] = profil_comportement(df_train[~tous_ctx])
display(pd.DataFrame(profils).T.round(1).rename_axis('Règle'))

,Nb clients,Mois sans paiement (moy.),% avec 6 mois sans paiement,% dette en hausse M-6 -> M-1,Utilisation plafond M-1 (médiane %),LIMIT_BAL (médiane)
Règle,,,,,,
R1_chronique,657.0,1.6,2.0,13.4,64.0,80000.0
R4_retard_6m,1095.0,2.3,11.4,17.2,63.6,60000.0
R5_retard_profond,949.0,2.9,13.0,36.6,66.4,50000.0
R6_retard_continu,1924.0,1.6,2.0,57.8,81.3,60000.0
R8_elargie,3209.0,1.9,5.2,42.3,72.3,60000.0
R1_chronique_bis,0.0,NaN,NaN,NaN,NaN,NaN
R4_retard_6m_bis,0.0,NaN,NaN,NaN,NaN,NaN
R5_retard_profond_bis,0.0,NaN,NaN,NaN,NaN,NaN
R6_retard_continu_bis,0.0,NaN,NaN,NaN,NaN,NaN


### A.3 Validation des règles sur le train

In [24]:
display(pd.DataFrame({nom: validation(df_train, masques_train[nom]) for nom in regles}).T.round(2).rename_axis('Règle'))

,Nb clients,Part (%),Taux défaut périmètre (%),Défauts captés (%),Taux défaut reste (%)
Règle,,,,,
R1_chronique,657.0,3.02,70.02,9.63,20.46
R4_retard_6m,1095.0,5.03,70.50,16.16,19.38
R5_retard_profond,949.0,4.36,62.49,12.41,20.10
R6_retard_continu,1924.0,8.84,52.34,21.08,19.00
R8_elargie,3209.0,14.75,58.18,39.08,15.69
R1_chronique_bis,0.0,0.00,NaN,0.00,21.95
R4_retard_6m_bis,0.0,0.00,NaN,0.00,21.95
R5_retard_profond_bis,0.0,0.00,NaN,0.00,21.95
R6_retard_continu_bis,0.0,0.00,NaN,0.00,21.95


In [25]:
# Synthèse compacte : base / bis / ter côte à côte pour chaque règle
synthese = pd.DataFrame({
    (version, indicateur): [
        (masques_train[nom + suffixe].sum() if indicateur == 'Nb clients'
         else df_train.loc[masques_train[nom + suffixe], 'dpnm'].mean() * 100)
        for nom in regles_base
    ]
    for version, suffixe in [('Base', ''), ('Bis', '_bis'), ('Ter', '_ter')]
    for indicateur in ['Nb clients', 'Taux défaut (%)']
}, index=list(regles_base)).round(1).rename_axis('Règle')
print("Synthèse base / bis / ter :")
display(synthese)

# Règles ter ventilées selon PAY_1 : ce que M-1 apporte réellement
categorie_pay1 = pd.cut(df_train['PAY_1'], bins=[-3, 0, 1, 9], labels=['PAY_1 <= 0', 'PAY_1 = 1', 'PAY_1 >= 2'])
ter_par_pay1 = pd.DataFrame({
    (cat, indicateur): [
        ((masques_train[nom + '_ter']) & (categorie_pay1 == cat)).sum() if indicateur == 'Nb clients'
        else df_train.loc[masques_train[nom + '_ter'] & (categorie_pay1 == cat), 'dpnm'].mean() * 100
        for nom in regles_base
    ]
    for cat in ['PAY_1 >= 2', 'PAY_1 = 1', 'PAY_1 <= 0']
    for indicateur in ['Nb clients', 'Taux défaut (%)']
}, index=list(regles_base)).round(1).rename_axis('Règle ter')
print("Règles ter ventilées selon la valeur de PAY_1 :")
display(ter_par_pay1)

Synthèse base / bis / ter :


Base                        Bis                  \
                  Nb clients Taux défaut (%) Nb clients Taux défaut (%)   
Règle                                                                     
R1_chronique             657            70.0          0             NaN   
R4_retard_6m            1095            70.5          0             NaN   
R5_retard_profond        949            62.5          0             NaN   
R6_retard_continu       1924            52.3          0             NaN   
R8_elargie              3209            58.2          0             NaN   

                         Ter                  
                  Nb clients Taux défaut (%)  
Règle                                         
R1_chronique             708            70.6  
R4_retard_6m            1118            70.1  
R5_retard_profond        798            61.3  
R6_retard_continu       1258            55.0  
R8_elargie              2527            61.3

Règles ter ventilées selon la valeur de PAY_1 :


PAY_1 >= 2                  PAY_1 = 1                  \
                  Nb clients Taux défaut (%) Nb clients Taux défaut (%)   
Règle ter                                                                 
R1_chronique             696            71.3          0             NaN   
R4_retard_6m            1095            70.5          0             NaN   
R5_retard_profond        619            66.1          0             NaN   
R6_retard_continu       1068            58.1          0             NaN   
R8_elargie              2190            64.6          0             NaN   

                  PAY_1 <= 0                  
                  Nb clients Taux défaut (%)  
Règle ter                                     
R1_chronique              12            33.3  
R4_retard_6m              23            52.2  
R5_retard_profond        179            44.7  
R6_retard_continu        190            37.9  
R8_elargie               337            39.5

## Annexe 3 : sortir d'un retard, et les retards d'avril

Combien faut-il payer pour sortir d'un retard, selon l'ancienneté du retard ? Et un 2 isolé en avril (`PAY_6` >= 2, `PAY_5` < 2) termine-t-il une série commencée en mars, mois que le dataset ne montre pas ? Étude sur le **train**, avec la lecture de l'audit : la codification `PAY_n` juge la facture `BILL_AMTn` sur le paiement du mois suivant `PAY_AMT(n-1)`.

### A3.1 Le paiement qui fait sortir d'un retard à 2
Deux populations, de juin à août : une **sortie après une suite de 2** (client à 2 deux mois de suite) et une **sortie après un seul 2** (client sain le mois d'avant le 2). Pour chacune : la part qui sort du retard (redevient -2, -1 ou 0) selon la part payée de la facture, puis selon le montant payé.

In [26]:
from IPython.display import Markdown

MOIS = {1: 'septembre', 2: 'août', 3: 'juillet', 4: 'juin', 5: 'mai', 6: 'avril'}
PARTS = ['rien payé', 'moins de 10 %'] + [f'{a} à {a + 10} %' for a in range(10, 100, 10)] + ['100 % et plus']
MONTANTS = ['rien payé', '1 à 500 NT$', '501 à 1 000 NT$', '1 001 à 1 500 NT$', '1 501 à 2 000 NT$', '2 001 à 3 000 NT$',
            '3 001 à 5 000 NT$', '5 001 à 10 000 NT$', 'plus de 10 000 NT$']

def sorties_apres_2(suite_de_2):
    # Sortie au mois n (juin à août) d'un client à 2 le mois précédent (PAY_(n+1) = 2) ;
    # suite de 2 : déjà à 2 le mois d'avant (PAY_(n+2) = 2) ; seul 2 : sain le mois d'avant (PAY_(n+2) <= 0)
    lignes = []
    for n in range(2, 5):
        m = (df_train[f'PAY_{n + 1}'] == 2) & (df_train[f'PAY_{n}'] != 1) & (df_train[f'BILL_AMT{n}'] > 0)
        m &= (df_train[f'PAY_{n + 2}'] == 2) if suite_de_2 else (df_train[f'PAY_{n + 2}'] <= 0)
        lignes.append(pd.DataFrame({'sort': df_train.loc[m, f'PAY_{n}'] <= 0, 'paiement': df_train.loc[m, f'PAY_AMT{n - 1}'],
                                    'facture': df_train.loc[m, f'BILL_AMT{n}']}))
    e = pd.concat(lignes, ignore_index=True)
    part = e['paiement'] / e['facture'] * 100
    e['part'] = pd.cut(part, [-0.001, 0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, np.inf], labels=PARTS, right=False).astype(object)
    e.loc[e['paiement'] <= 0, 'part'] = 'rien payé'
    e['montant'] = pd.cut(e['paiement'], [-0.001, 0, 500, 1000, 1500, 2000, 3000, 5000, 10000, np.inf], labels=MONTANTS)
    return e

def tableau(e, colonne, ordre, titre):
    g = e.groupby(colonne)['sort'].agg(Factures='size', Sortent=lambda x: round(x.mean() * 100, 1)).reindex(ordre)
    g.columns = ['Factures', 'Sortent du retard (%)']
    g.index.name = titre
    display(g)

for libelle, suite_de_2 in [('Sortie après une suite de 2', True), ('Sortie après un seul 2', False)]:
    e = sorties_apres_2(suite_de_2)
    display(Markdown(f'**{libelle}** (juin à août)'))
    tableau(e, 'part', PARTS, 'Part payée de la facture')
    tableau(e, 'montant', MONTANTS, 'Montant payé')


**Sortie après une suite de 2** (juin à août)

,Factures,Sortent du retard (%)
Part payée de la facture,,
rien payé,1307,0.1
moins de 10 %,2250,13.6
10 à 20 %,567,43.0
20 à 30 %,126,47.6
30 à 40 %,46,52.2
40 à 50 %,32,50.0
50 à 60 %,20,15.0
60 à 70 %,5,60.0
70 à 80 %,10,40.0


,Factures,Sortent du retard (%)
Montant payé,,
rien payé,1307,0.1
1 à 500 NT$,119,9.2
501 à 1 000 NT$,232,12.9
1 001 à 1 500 NT$,273,7.0
1 501 à 2 000 NT$,503,11.9
2 001 à 3 000 NT$,602,25.7
3 001 à 5 000 NT$,718,32.5
5 001 à 10 000 NT$,501,29.1
plus de 10 000 NT$,225,48.9


**Sortie après un seul 2** (juin à août)

,Factures,Sortent du retard (%)
Part payée de la facture,,
rien payé,365,0.5
moins de 10 %,1201,34.1
10 à 20 %,622,60.9
20 à 30 %,189,68.8
30 à 40 %,78,70.5
40 à 50 %,44,63.6
50 à 60 %,46,54.3
60 à 70 %,13,38.5
70 à 80 %,15,53.3


,Factures,Sortent du retard (%)
Montant payé,,
rien payé,365,0.5
1 à 500 NT$,88,15.9
501 à 1 000 NT$,184,35.9
1 001 à 1 500 NT$,193,29.0
1 501 à 2 000 NT$,292,33.6
2 001 à 3 000 NT$,457,57.1
3 001 à 5 000 NT$,608,59.0
5 001 à 10 000 NT$,437,66.8
plus de 10 000 NT$,279,77.1


**Comment lire** : quatre tableaux, deux par population (sortie après une suite de 2, puis après un seul 2) ; dans chacun, une ligne par tranche, le nombre de factures et la part des clients qui sortent du retard.

**Interprétation**  
Sans paiement, on ne sort jamais d'un 2 ; en soldant la facture, presque toujours. Entre les deux, la sortie progresse avec la part payée et avec le montant payé.  
Après une suite de 2, il faut payer nettement plus pour sortir qu'après un seul 2, à part payée ou à montant égal : l'arriéré à rattraper est plus grand.  
Les tranches de 50 à 100 % ont de petits effectifs.

### A3.2 Les 2 isolés d'avril : une série commencée en mars ?
La codification de mars n'est pas dans le dataset. Deux indices permettent pourtant de la deviner :
- un retard ne monte que d'un cran par mois : un `PAY_6` à 3 ou plus prouve que mars était déjà en retard ;
- la codification de mars jugerait la facture de mars sur le paiement d'avril (`PAY_AMT6`), connu ; seul le montant de la facture de mars manque.

Je mesure d'abord, sur les mois où tout est visible (mai à août), si le mois d'avant d'un 2 était déjà en retard selon le paiement du mois, puis je classe les 2 isolés d'avril.

In [27]:
# 1. Client à 2 au mois n (mai à août) : le mois d'avant était-il déjà en retard, selon le paiement du mois (montant seul, comme en avril) ?
lignes = []
for n in range(5, 1, -1):
    m = df_train[f'PAY_{n}'] == 2
    lignes.append(pd.DataFrame({'mois': MOIS[n], 'avant_en_retard': df_train.loc[m, f'PAY_{n + 1}'] >= 2, 'paiement': df_train.loc[m, f'PAY_AMT{n}']}))
deux = pd.concat(lignes, ignore_index=True)
deux['montant'] = pd.cut(deux['paiement'], [-0.001, 0, 500, 1000, 2000, 5000, np.inf],
                         labels=['rien', '1 à 500 NT$', '501 à 1 000 NT$', '1 001 à 2 000 NT$', '2 001 à 5 000 NT$', 'plus de 5 000 NT$'])
ordre_mois = [MOIS[n] for n in range(5, 1, -1)]
g = deux.groupby(['montant', 'mois'], observed=False)['avant_en_retard']
display(Markdown("**Client à 2 : part dont le mois d'avant était déjà en retard (%)**, selon le paiement du mois, puis nombre de clients"))
display((g.mean() * 100).round(1).unstack()[ordre_mois].rename_axis(index='Paiement du mois', columns='Mois'))
display(g.size().unstack()[ordre_mois].rename_axis(index='Paiement du mois', columns='Mois'))

# 2. Les 2 isolés d'avril (PAY_6 >= 2, PAY_5 < 2), classés selon les deux indices
iso = (df_train['PAY_6'] >= 2) & (df_train['PAY_5'] < 2)
classes = pd.Series(np.select(
    [df_train['PAY_6'] >= 3, df_train['PAY_AMT6'] == 0],
    ['PAY_6 à 3 ou plus : série certaine', 'PAY_6 = 2, rien payé en avril : série très probable'],
    'PAY_6 = 2, paiement en avril : indécidable'), index=df_train.index)[iso]
display(classes.value_counts().rename_axis("2 isolés d'avril").to_frame('Clients'))


**Client à 2 : part dont le mois d'avant était déjà en retard (%)**, selon le paiement du mois, puis nombre de clients

Mois,mai,juin,juillet,août
Paiement du mois,,,,
rien,94.8,87.4,85.5,95.5
1 à 500 NT$,45.1,36.3,42.9,64.2
501 à 1 000 NT$,39.7,33.7,40.6,59.4
1 001 à 2 000 NT$,59.7,55.0,43.7,34.6
2 001 à 5 000 NT$,63.9,60.9,55.0,60.6
plus de 5 000 NT$,67.9,57.6,48.2,49.9


Mois,mai,juin,juillet,août
Paiement du mois,,,,
rien,555,692,855,864
1 à 500 NT$,113,157,191,162
501 à 1 000 NT$,237,332,342,165
1 001 à 2 000 NT$,457,491,627,691
2 001 à 5 000 NT$,496,540,635,824
plus de 5 000 NT$,249,288,367,453


,Clients
2 isolés d'avril,
"PAY_6 = 2, paiement en avril : indécidable",600
"PAY_6 = 2, rien payé en avril : série très probable",108
PAY_6 à 3 ou plus : série certaine,32


**Comment lire** : premier tableau, pour un client à 2 de mai à août, la part dont le mois d'avant était déjà en retard, selon ce qu'il a payé dans le mois ; puis les 2 isolés d'avril classés selon les deux indices.

**Interprétation**
Quand un client à 2 n'a rien payé dans le mois, le mois d'avant était presque toujours déjà en retard ; quand il a payé quelque chose, c'est une chance sur deux environ, et le montant ne départage pas (la facture qu'il règle n'est pas connue).
Pour les 2 isolés d'avril : un `PAY_6` à 3 ou plus est une série certaine (passage en incident sans discussion) ; un 2 sans paiement en avril est très probablement une série ; un 2 avec un paiement reste indécidable.